In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import pandas as pd
from utils import load_data, check_stationarity
from sklearn.metrics import mean_absolute_error, mean_squared_error
from models import *
from statsmodels.stats.diagnostic import acorr_ljungbox

import warnings
warnings.filterwarnings("ignore", message="X does not have valid feature names")

global_data = load_data()

In [ ]:
def evaluate(y_test, y_pred):
    print("MAE:", mean_absolute_error(y_test, y_pred))
    print("RMSE:", np.sqrt(mean_squared_error(y_test, y_pred)))

    y_test = np.array(y_test)
    y_pred = np.array(y_pred)

    residuals = y_test - y_pred
    std = np.std(residuals, ddof=1)

    print("Standard Deviation of Residuals:", std)

In [ ]:
from statsmodels.graphics.tsaplots import plot_acf, plot_pacf
from plots import plot_lb_test

def plots_from_results(results, lb_results):
    # Preiction vs Actual plot
    plt.figure(figsize=(12, 6))
    plt.plot(results_df["ds"], results_df["actual"], label="Actual", linewidth=2)
    plt.plot(results_df["ds"], results_df["predicted"], label="Predicted", linewidth=2)
    plt.title("Actual vs Predicted values")
    plt.xlabel("Date")
    plt.ylabel("Value")
    plt.legend()
    plt.grid(True)
    plt.show()

    residuals = np.array(results.actual) - np.array(results.predicted)

    plt.figure(figsize=(12, 4))
    plt.scatter(results_df["ds"], residuals)
    plt.axhline(0)
    plt.title("Residuals over time")
    plt.xlabel("Date")
    plt.ylabel("Residual")
    plt.show()

    lb_df = pd.DataFrame(lb_results)
    plot_lb_test(lb_df)


    fig, axes = plt.subplots(1, 2, figsize=(12, 4))

    plot_acf(residuals, lags=12, ax=axes[0])
    axes[0].set_title("ACF of Residuals")
    
    #plot_pacf(residuals, lags=20, ax=axes[1], method="ywm")
    #axes[1].set_title("PACF of Residuals")
    
    plt.tight_layout()
    plt.show()


In [ ]:
def recursive_forecast(model, data, n_steps=12, n_lags=12):
    """
    Generate recursive multi-step forecasts for monthly data using a trained model.
    
    Parameters:
    - model: trained ML model (e.g., XGBoost)
    - data: pd.DataFrame with a 'y' column and datetime index
    - n_steps: number of months to forecast
    - n_lags: number of lag features to use
    
    Returns:
    - forecast: pd.Series of predicted values with datetime index
    """
    last_data = data.copy()
    forecast = []
    
    for i in range(n_steps):
        # Compute lag features
        lags = [last_data['y'].iloc[-lag] for lag in range(1, n_lags + 1)]
        
        # Rolling statistics
        rolling_3 = np.mean([last_data['y'].iloc[-j] for j in range(1, min(4, len(last_data)+1))])
        rolling_6 = np.mean([last_data['y'].iloc[-j] for j in range(1, min(7, len(last_data)+1))])
        
        # Month features
        next_month = last_data.index[-1] + pd.DateOffset(months=1)
        month_sin = np.sin(2 * np.pi * next_month.month / 12)
        month_cos = np.cos(2 * np.pi * next_month.month / 12)
        
        # Combine features
        X_next = np.array(lags + [rolling_3, rolling_6, next_month.month, month_sin, month_cos]).reshape(1, -1)
        
        # Predict next value
        y_next = model.predict(X_next)[0]
        forecast.append(y_next)
        
        # Append prediction for next iteration
        last_data = pd.concat([last_data, pd.DataFrame({'y': [y_next]}, index=[next_month])])
    
    forecast_index = pd.date_range(start=last_data.index[-n_steps], periods=n_steps, freq='MS')
    return pd.Series(forecast, index=forecast_index)

    

In [ ]:
def get_data(target):
    loaded_data = global_data
    data = pd.DataFrame({
        'date': loaded_data['date'],
        'y': loaded_data[target]
    })
    
    data.set_index('date', inplace=True)
    
    # Add lag features (1 to 12 months)
    for lag in range(1, 13):
        data[f'lag_{lag}'] = data['y'].shift(lag)
    
    # Add rolling mean features
    data['rolling_3'] = data['y'].shift(1).rolling(3).mean()
    data['rolling_6'] = data['y'].shift(1).rolling(6).mean()
    
    # Add cyclical month features
    data['month'] = data.index.month
    data['month_sin'] = np.sin(2 * np.pi * data['month']/12)
    data['month_cos'] = np.cos(2 * np.pi * data['month']/12)
    
    # Drop rows with NaN from lag/rolling features
    data.dropna(inplace=True)

    return data

In [ ]:
def get_model_prediction(model, train, steps):
    X_train = train.drop(columns=['y'])
    y_train = train['y']

    model.fit(X_train, y_train)
    return recursive_forecast(model, train, n_steps=steps)

In [ ]:
"""
split_idx = int(len(global_data)*0.8)
max_data = get_data('tmax')
min_data = get_data('tmin')
test_diff = max_data.iloc[split_idx:]['y'] - min_data.iloc[split_idx:]['y']
"""

In [ ]:
import itertools

param_grid_rf = {
    "n_estimators": [200, 500],
    "max_depth": [None, 10, 20],
    "min_samples_leaf": [1, 5, 10],
    "max_features": ["sqrt", 0.5]
}

param_grid_xgb = {
    "learning_rate": [0.01, 0.05, 0.1],
    "max_depth": [3, 5, 7],
    "min_child_weight": [1, 5, 10],
    "subsample": [0.7, 0.9],
    "colsample_bytree": [0.7, 0.9]
}



In [ ]:
from sklearn.ensemble import RandomForestRegressor
from xgboost import XGBRegressor

window_size = 96  # 8 years
forecast_horizon = 24  # 2-year ahead

def sliding_window_rf(params):
    preds = []
    actuals = []
    dates = []
    lb_results = []
    rmse_per_fold = []

    max_data = get_data('tmax')
    min_data = get_data('tmin')
    diff_data = get_data('tdiff')

    for start in range(0, len(diff_data) - window_size - forecast_horizon + 1, forecast_horizon):
        # Training and test windows
        max_train_window = max_data.iloc[start:start + window_size]
        min_train_window = min_data.iloc[start:start + window_size]
        diff_test_window = diff_data.iloc[start + window_size:start + window_size + forecast_horizon]

        
        rf_max = RandomForestRegressor(
            **params,
            random_state=42,
            n_jobs=-1
        )
        
        rf_min = RandomForestRegressor(
            **params,
            random_state=42,
            n_jobs=-1
        )

        max_pred_rf = get_model_prediction(rf_max, max_train_window, forecast_horizon)
        min_pred_rf = get_model_prediction(rf_min, min_train_window, forecast_horizon)
        diff_pred_rf = max_pred_rf - min_pred_rf

        preds.extend(diff_pred_rf.values)
        actuals.extend(diff_test_window['y'].values)
        dates.extend(diff_test_window.index)

        residuals = diff_test_window['y'].values - diff_pred_rf.values
        rmse = np.sqrt(mean_squared_error(diff_test_window['y'].values, diff_pred_rf.values))
        rmse_per_fold.append(rmse)

        # Ljung–Box at seasonal lag
        lb = acorr_ljungbox(
            residuals,
            lags=[12],
            return_df=True
        )

        lb_results.append({
            'ds': diff_test_window.index[-1],   # end of forecast window
            'lb_stat': lb['lb_stat'].iloc[0],
            'p_value': lb['lb_pvalue'].iloc[0]
        })

    rmse_mean = np.mean(rmse_per_fold)
    rmse_std = np.std(rmse_per_fold, ddof=1)  # sample std

    #DataFrame for plotting
    results_df = pd.DataFrame({
        'ds': dates,
        'actual': actuals,
        'predicted': preds
    })

    return -rmse_mean, rmse_std, results_df, lb_results


def sliding_window_xgb(params):
    preds = []
    actuals = []
    dates = []
    lb_results = []
    rmse_per_fold = []

    max_data = get_data('tmax')
    min_data = get_data('tmin')
    diff_data = get_data('tdiff')

    for start in range(0, len(diff_data) - window_size - forecast_horizon + 1, forecast_horizon):
        # Training and test windows
        max_train_window = max_data.iloc[start:start + window_size]
        min_train_window = min_data.iloc[start:start + window_size]
        diff_test_window = diff_data.iloc[start + window_size:start + window_size + forecast_horizon]

        
        rf_max = XGBRegressor(
            **params,
            random_state=42,
            n_jobs=-1
        )
        
        rf_min = XGBRegressor(
            **params,
            random_state=42,
            n_jobs=-1
        )

        max_pred_rf = get_model_prediction(rf_max, max_train_window, forecast_horizon)
        min_pred_rf = get_model_prediction(rf_min, min_train_window, forecast_horizon)
        diff_pred_rf = max_pred_rf - min_pred_rf

        preds.extend(diff_pred_rf.values)
        actuals.extend(diff_test_window['y'].values)
        dates.extend(diff_test_window.index)

        residuals = diff_test_window['y'].values - diff_pred_rf.values
        rmse = np.sqrt(mean_squared_error(diff_test_window['y'].values, diff_pred_rf.values))
        rmse_per_fold.append(rmse)

        # Ljung–Box at seasonal lag
        lb = acorr_ljungbox(
            residuals,
            lags=[12],
            return_df=True
        )

        lb_results.append({
            'ds': diff_test_window.index[-1],   # end of forecast window
            'lb_stat': lb['lb_stat'].iloc[0],
            'p_value': lb['lb_pvalue'].iloc[0]
        })

    rmse_mean = np.mean(rmse_per_fold)
    rmse_std = np.std(rmse_per_fold, ddof=1)  # sample std

    #DataFrame for plotting
    results_df = pd.DataFrame({
        'ds': dates,
        'actual': actuals,
        'predicted': preds
    })

    return -rmse_mean, rmse_std, results_df, lb_results


In [ ]:
best_score = float("-inf")
best_params_rf = None
results = []
results_df = pd.DataFrame()
results_lb = []

keys, values = zip(*param_grid_rf.items())

print("Running...")
for combo in itertools.product(*values):
    params = dict(zip(keys, combo))

    score, std, new_results_df, new_results_lb = sliding_window_rf(params)

    results.append((params, score))

    if score > best_score:
        best_score = score
        best_params_rf = params
        results_df = new_results_df
        results_lb = new_results_lb

print(f"Params: {best_params_rf} → Score: {-best_score:.4f}")
plots_from_results(results_df, results_lb)

In [ ]:
best_score = float("-inf")
best_params_xgb = None
results = []
results_df = pd.DataFrame()
results_lb = []

keys, values = zip(*param_grid_xgb.items())

for combo in itertools.product(*values):
    params = dict(zip(keys, combo))

    score, std, new_results_df, new_results_lb = sliding_window_xgb(params)

    results.append((params, score))

    if score > best_score:
        best_score = score
        best_params_xgb = params
        results_df = new_results_df
        results_lb = new_results_lb

print(f"Params: {best_params_xgb} → Score: {-best_score:.4f}")
plots_from_results(results_df, results_lb)

In [ ]:
forecast_horizon = 24  # 2-year ahead

preds = []
actuals = []
dates = []
lb_results = []

max_data = get_data('tmax')
min_data = get_data('tmin')
diff_data = get_data('tdiff')

train_max = max_data.iloc[:-forecast_horizon]
train_min = min_data.iloc[:-forecast_horizon]
test_diff = diff_data.iloc[-forecast_horizon:]

rf_max = RandomForestRegressor(
    **best_params_rf,
    random_state=42,
    n_jobs=-1
)
        

rf_min = RandomForestRegressor(
    **best_params_rf,
    random_state=42,
    n_jobs=-1
)
      

max_pred_rf = get_model_prediction(rf_max, train_max, forecast_horizon)
min_pred_rf = get_model_prediction(rf_min, train_min, forecast_horizon)
diff_pred_rf = max_pred_rf - min_pred_rf

preds.extend(diff_pred_rf.values)
actuals.extend(test_diff['y'].values)
dates.extend(test_diff.index)

residuals = test_diff['y'].values - diff_pred_rf.values

# Ljung–Box at seasonal lag
lb = acorr_ljungbox(
    residuals,
    lags=[12],
    return_df=True
)

lb_results.append({
    'ds': test_diff.index[-1],   # end of forecast window
    'lb_stat': lb['lb_stat'].iloc[0],
    'p_value': lb['lb_pvalue'].iloc[0]
})

evaluate(actuals, preds)
#DataFrame for plotting
results_df = pd.DataFrame({
    'ds': dates,
    'actual': actuals,
    'predicted': preds
})

plots_from_results(results_df, lb_results)

In [ ]:
forecast_horizon = 24  # 2-year ahead

preds = []
actuals = []
dates = []
lb_results = []

max_data = get_data('tmax')
min_data = get_data('tmin')
diff_data = get_data('tdiff')

train_max = max_data.iloc[:-forecast_horizon]
train_min = min_data.iloc[:-forecast_horizon]
test_diff = diff_data.iloc[-forecast_horizon:]

rf_max = XGBRegressor(
    **best_params_xgb,
    random_state=42,
    n_jobs=-1
)

rf_min = XGBRegressor(
    **best_params_xgb,
    random_state=42,
    n_jobs=-1
)

max_pred_rf = get_model_prediction(rf_max, train_max, forecast_horizon)
min_pred_rf = get_model_prediction(rf_min, train_min, forecast_horizon)
diff_pred_rf = max_pred_rf - min_pred_rf

preds.extend(diff_pred_rf.values)
actuals.extend(test_diff['y'].values)
dates.extend(test_diff.index)

residuals = test_diff['y'].values - diff_pred_rf.values

# Ljung–Box at seasonal lag
lb = acorr_ljungbox(
    residuals,
    lags=[12],
    return_df=True
)

lb_results.append({
    'ds': test_diff.index[-1],   # end of forecast window
    'lb_stat': lb['lb_stat'].iloc[0],
    'p_value': lb['lb_pvalue'].iloc[0]
})

evaluate(actuals, preds)
#DataFrame for plotting
results_df = pd.DataFrame({
    'ds': dates,
    'actual': actuals,
    'predicted': preds
})

plots_from_results(results_df, lb_results)